# Original coursework: sanitized publication copy

This is a sanitized copy of the recovered original course notebook, not a newly reconstructed demo. Outputs, execution counts and incidental metadata were cleared before publication to avoid exposing customer review excerpts, local paths and account information. The source has not been rerun here against its original dataset. Team credits and original AI-use disclosure remain. Any credentials must be supplied privately. The recovered hotel notebook had no saved execution counts. No verified original numeric performance is inferred from it.


In [ ]:
# =========================================
# IMPORT LIBRARIES
# =========================================

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning - Model building
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)

# Other models
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

# Plot styling (optional but makes graphs look better)
sns.set(style="whitegrid")

# Predicting Hotel Booking Cancellations using Machine Learning

## 1. Problem Overview

In the hospitality industry, booking cancellations represent a significant challenge for revenue management and operational planning. Hotels rely on accurate demand forecasts to optimize room pricing, staffing, and inventory allocation. However, unexpected cancellations lead to lost revenue and inefficient resource utilization.

This project aims to build machine learning models to predict whether a hotel booking will be canceled using historical booking data. By identifying high-risk bookings in advance, hotels can take proactive actions such as requiring deposits, adjusting overbooking strategies, or targeting customers with incentives.

## 2. Importance of the Problem

Accurate cancellation prediction helps:
- Improve revenue management strategies
- Reduce financial losses from no-shows
- Optimize staffing and resource allocation
- Enhance customer segmentation and targeting

## 3. Stakeholders

The key stakeholders include:
- Hotel revenue managers
- Operations and front desk teams
- Marketing teams
- Customers (indirectly through better service and pricing strategies)

## 4. Dataset Overview

The dataset contains 119,390 hotel booking records. Each record represents a reservation with information about customer demographics, booking details, and hotel characteristics.

The target variable is:
- **is_canceled** (1 = canceled, 0 = not canceled)

Before building models, it is important to understand the data structure and identify potential data quality issues.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/hotel_bookings.csv')
df.head()

## 5. Initial Data Exploration

Before cleaning and modeling, it is essential to explore the dataset to understand its structure, variable types, and potential data quality issues.

This step helps identify:
- The number of observations and features
- Data types (numerical vs categorical)
- Presence of missing values
- Potential inconsistencies in the dataset

In [ ]:
# Check dataset shape (rows, columns)
df.shape

In [ ]:
# Check data types and missing values
df.info()

In [ ]:
# View first rows
df.head()

The dataset contains a large number of booking records with a mix of numerical and categorical variables.

Initial inspection shows that some variables contain missing values, which will need to be handled before building machine learning models. Additionally, the dataset includes various features related to customer behavior, booking characteristics, and hotel operations, which are expected to influence cancellation behavior.

## 6. Data Cleaning

To ensure the quality and reliability of the analysis, missing values are handled using a structured approach based on the assignment requirements.

- Columns with a large number of missing values (greater than 500 observations) are removed to avoid introducing bias.
- Rows with a small number of missing values are removed to preserve data integrity while minimizing data loss.

In [ ]:
# Check missing values
missing_values = df.isnull().sum()
missing_values.sort_values(ascending=False)

In [ ]:
# Drop columns with more than 500 missing values
cols_to_drop = missing_values[missing_values > 500].index
df = df.drop(columns=cols_to_drop)

In [ ]:
# Drop remaining rows with missing values
df = df.dropna()

In [ ]:
# Check again
df.isnull().sum().sum()

Columns with a large number of missing values were removed to prevent bias and unreliable patterns in the model. For the remaining variables, rows with missing values were removed to maintain consistency in the dataset.

After cleaning, the dataset is complete and ready for further analysis and modeling.

## 7. Data Visualization and Exploratory Analysis

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x='is_canceled', data=df)
plt.title("Distribution of Booking Cancellations")
plt.show()

The distribution of the target variable shows that a significant proportion of bookings are canceled. This highlights the importance of predicting cancellations accurately, as it directly impacts hotel revenue and operations.

In [ ]:
sns.boxplot(x='is_canceled', y='lead_time', data=df)
plt.title("Lead Time vs Cancellation")
plt.show()

Bookings with longer lead times tend to have a higher likelihood of cancellation. This suggests that customers who book far in advance may be more uncertain and therefore more likely to cancel.

In [ ]:
sns.countplot(x='deposit_type', hue='is_canceled', data=df)
plt.xticks(rotation=45)
plt.show()

Customers who do not make a deposit are more likely to cancel their bookings. This indicates that financial commitment plays an important role in reducing cancellation risk.

In [ ]:
sns.countplot(x='is_repeated_guest', hue='is_canceled', data=df)
plt.show()

Repeat customers are significantly less likely to cancel their bookings, suggesting that customer loyalty is associated with more reliable booking behavior.

## 8. Feature Selection

Before building machine learning models, it is important to carefully select the features used for prediction.

Not all variables are appropriate for modeling. Some variables may introduce data leakage, meaning they contain information that would not be available at the time of prediction.

For example:
- reservation_status and reservation_status_date directly reflect the outcome of the booking and therefore must be removed.

Including such variables would artificially inflate model performance and reduce the model’s real-world applicability.

Additionally, categorical variables are converted into numerical format using one-hot encoding to ensure compatibility with machine learning algorithms.

In [ ]:
# Remove leakage columns
df = df.drop(columns=['reservation_status', 'reservation_status_date'], errors='ignore')

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

After removing data leakage variables and encoding categorical features, the dataset is prepared for modeling.

The data is split into training and testing sets to ensure that model performance is evaluated on unseen data, which provides a more realistic estimate of how the model will perform in real-world scenarios.

## 9. Decision Tree Model

A Decision Tree classifier is used as the primary model due to its interpretability and ability to capture non-linear relationships between features.

To ensure the model remains interpretable, the tree depth is constrained, allowing the structure of the tree to be visualized and analyzed.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)

In [ ]:
y_pred = dt.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

The decision tree model demonstrates strong performance in predicting booking cancellations.

The balance between precision and recall indicates that the model is effective in identifying both canceled and non-canceled bookings, which is essential for operational decision-making.

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(20,10))
plot_tree(dt, filled=True, feature_names=X.columns)
plt.show()

In [ ]:
import pandas as pd

importances = pd.Series(dt.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(10).plot(kind='bar')
plt.title("Top 10 Important Features")
plt.show()

The feature importance results show that variables such as lead time, deposit type, and customer history play a significant role in predicting cancellations.

These insights are valuable for business decision-making, as they highlight the key drivers behind customer behavior.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

y_probs = dt.predict_proba(X_test)[:,1]

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_probs)
plt.plot(fpr, tpr)
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_probs)
plt.plot(recall, precision)
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(dt, X, y, cv=10)
print("Average Accuracy:", cv_scores.mean())

## 10. Pre-Pruning Optimization

To improve model performance and prevent overfitting, pre-pruning techniques are applied to the decision tree model.

Pre-pruning controls the growth of the tree by limiting its complexity before it fully develops. This helps the model generalize better to unseen data.

Grid Search with cross-validation is used to identify the optimal combination of hyperparameters.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [5, 7, 10],
    'max_leaf_nodes': [10, 20, 30, 40]
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

In [ ]:
results = pd.DataFrame(grid.cv_results_)

pivot = results.pivot(
    index='param_max_depth',
    columns='param_max_leaf_nodes',
    values='mean_test_score'
)

import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(pivot, annot=True, cmap='coolwarm')
plt.title("Grid Search Performance")
plt.show()

The heatmap illustrates how model accuracy varies across different combinations of tree depth and number of leaf nodes.

The selected parameters represent the best balance between model complexity and predictive performance, ensuring the model does not overfit while maintaining high accuracy.

In [ ]:
best_dt = grid.best_estimator_

y_pred_best = best_dt.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("Precision:", precision_score(y_test, y_pred_best))
print("Recall:", recall_score(y_test, y_pred_best))
print("F1 Score:", f1_score(y_test, y_pred_best))

In [ ]:
sns.heatmap(confusion_matrix(y_test, y_pred_best), annot=True, fmt='d')
plt.title("Confusion Matrix - Pruned Tree")
plt.show()

In [ ]:
importances = pd.Series(best_dt.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(10).plot(kind='bar')
plt.title("Top Features (Pruned Tree)")
plt.show()

## 11. Model Comparison

To evaluate the effectiveness of the decision tree model, additional models are implemented:

- K-Nearest Neighbors (KNN), which captures local patterns in the data
- Logistic Regression, which models linear relationships between features and the target variable

These models provide a benchmark for comparison.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)

In [ ]:
print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print("KNN Precision:", precision_score(y_test, y_pred_knn))
print("KNN Recall:", recall_score(y_test, y_pred_knn))
print("KNN F1:", f1_score(y_test, y_pred_knn))

In [ ]:
from sklearn.linear_model import LogisticRegression

log = LogisticRegression(max_iter=1000)
log.fit(X_train, y_train)

y_pred_log = log.predict(X_test)

In [ ]:
print("Logistic Accuracy:", accuracy_score(y_test, y_pred_log))
print("Logistic Precision:", precision_score(y_test, y_pred_log))
print("Logistic Recall:", recall_score(y_test, y_pred_log))
print("Logistic F1:", f1_score(y_test, y_pred_log))

## 12. Conclusion

### Best Model Performance

Among the models tested, the pre-pruned decision tree achieved the best overall performance in terms of accuracy and precision. This indicates that the model is highly effective in correctly identifying booking cancellations while minimizing false predictions.

KNN demonstrated moderate performance but was less efficient due to its sensitivity to feature scaling and dataset size. Logistic regression provided stable results but was limited in capturing complex non-linear relationships in the data.

### Model Selection Criteria

The final model was selected based on:
- Accuracy: overall correctness of predictions
- Precision: ability to correctly identify cancellations
- Recall: ability to capture actual cancellations

Given the business context, precision and recall are both important, as hotels need to correctly identify cancellations without misclassifying reliable customers.

### Managerial Implications

The decision tree model provides actionable insights for hotel management:

- Customers with longer lead times are more likely to cancel
- Bookings without deposits have a higher cancellation risk
- Repeat guests are less likely to cancel

These insights can help managers:
- Implement targeted deposit policies
- Optimize overbooking strategies
- Improve customer segmentation and retention strategies

In [ ]:
!jupyter nbconvert --to html YourNotebookName.ipynb

In [ ]:
!jupyter nbconvert --to html "/content/Welcome To Colab.ipynb"

In [ ]:
!ls /content

In [ ]:
!ls /content/drive/MyDrive

In [ ]:
!ls /content/drive/MyDrive/Colab\ Notebooks

In [ ]:
!jupyter nbconvert --to html "/content/drive/MyDrive/Colab Notebooks/YourNotebookName.ipynb"